In [ ]:
# ==================== STAGE 8 (FINAL FIXED) ====================
# Smart Tarannum Training System
# Streamlit + CRNN (ONNX) + Arduino LED Output
# - Record / Upload / Samples
# - Upload replay
# - Arduino connection stable across reruns (session_state)
# ===============================================================

import streamlit as st
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import onnxruntime as ort
import soundfile as sf
import tempfile
import time
import os

# --- PySerial (make sure installed: pip install pyserial) ---
import serial
from serial.tools import list_ports

# ================= CONFIG =================
SR = 22050
N_MELS = 128
N_FFT = 2048
HOP = 512
FMIN = 30
FMAX = 8000
T_TARGET = 430

CLASSES = ["Hijaz", "Nahawand", "Saba"]
ONNX_PATH = r"C:\Users\nabal\Documents\FYP\models_deep\crnn_mel_best.onnx"

# Default port (still editable via dropdown)
DEFAULT_PORT = "COM8"
BAUD = 9600

# Rejection thresholds
THR_CONF = 0.75
THR_MARGIN = 0.20
THR_ENTROPY = 0.90

# ================= STREAMLIT UI =================
st.set_page_config(page_title="Smart Tarannum Training System", layout="centered")
st.title("🎵 Smart Tarannum Training System")
st.caption("Real-Time Maqām Classification, Confidence Scoring & Arduino LED Output")

# ================= SESSION STATE =================
if "uploaded_path" not in st.session_state:
    st.session_state.uploaded_path = None

if "input_source" not in st.session_state:
    st.session_state.input_source = None

if "ser" not in st.session_state:
    st.session_state.ser = None

if "selected_port" not in st.session_state:
    st.session_state.selected_port = DEFAULT_PORT

# ================= MODEL LOAD =================
@st.cache_resource
def load_model():
    return ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

session = load_model()

# ================= ARDUINO UTILITIES =================
def available_ports():
    ports = [p.device for p in list_ports.comports()]
    return ports

def close_serial():
    try:
        if st.session_state.ser is not None and st.session_state.ser.is_open:
            st.session_state.ser.close()
    except Exception:
        pass
    st.session_state.ser = None

def connect_serial(port, baud=9600):
    """
    Rerun-safe connector:
    - closes any old handle
    - reopens fresh
    """
    close_serial()
    try:
        s = serial.Serial(port, baud, timeout=1)
        time.sleep(2)  # allow Arduino reset
        st.session_state.ser = s
        return True, None
    except Exception as e:
        st.session_state.ser = None
        return False, str(e)

def ensure_serial_ok():
    """
    If connection exists but is closed/stale, try to reopen using selected port.
    """
    s = st.session_state.ser
    if s is None:
        return False
    try:
        if not s.is_open:
            ok, _ = connect_serial(st.session_state.selected_port, BAUD)
            return ok
        return True
    except Exception:
        # if any error, drop and reconnect
        ok, _ = connect_serial(st.session_state.selected_port, BAUD)
        return ok

def send_cmd(cmd_byte: bytes):
    """
    Send one-byte command: b'H', b'N', b'S', b'U'
    """
    if not ensure_serial_ok():
        return False, "Arduino not connected"
    try:
        st.session_state.ser.write(cmd_byte)
        st.session_state.ser.flush()
        return True, None
    except Exception as e:
        # attempt one reconnect and retry
        ok, _ = connect_serial(st.session_state.selected_port, BAUD)
        if not ok:
            return False, f"Write failed; reconnect failed: {e}"
        try:
            st.session_state.ser.write(cmd_byte)
            st.session_state.ser.flush()
            return True, None
        except Exception as e2:
            return False, f"Write failed after reconnect: {e2}"

# ================= ARDUINO PANEL =================
st.subheader("🔌 Arduino Connection")

ports = available_ports()
if len(ports) == 0:
    st.warning("No COM ports detected. Plug in Arduino and install driver if needed.")
else:
    # keep previous selection if still exists
    if st.session_state.selected_port not in ports:
        st.session_state.selected_port = ports[0]

    st.session_state.selected_port = st.selectbox(
        "Select Arduino COM Port",
        options=ports,
        index=ports.index(st.session_state.selected_port) if st.session_state.selected_port in ports else 0
    )

colA, colB = st.columns(2)
with colA:
    if st.button("Connect / Reconnect Arduino"):
        ok, err = connect_serial(st.session_state.selected_port, BAUD)
        if ok:
            st.success(f"Arduino connected on {st.session_state.selected_port}")
        else:
            st.error(f"Failed to connect on {st.session_state.selected_port}: {err}")

with colB:
    if st.button("Disconnect Arduino"):
        close_serial()
        st.warning("Arduino disconnected")

if ensure_serial_ok():
    st.success("🟢 Arduino status: Connected (LED output enabled)")
else:
    st.warning("⚠️ Arduino status: Not connected (LED output disabled)")

st.divider()

st.subheader("🔄 System Control")

if st.button("Reset System"):
    # Clear audio & state
    st.session_state.uploaded_path = None
    st.session_state.input_source = None

    # Turn OFF all LEDs
    if ensure_serial_ok():
        send_cmd(b'U')  # or define a special OFF command if you want

    st.success("System reset. Ready for new input.")
    st.experimental_rerun()


# ================= AUDIO → MEL =================
def wav_to_mel(path):
    y, _ = librosa.load(path, sr=SR, mono=True)

    S = librosa.feature.melspectrogram(
        y=y, sr=SR,
        n_fft=N_FFT, hop_length=HOP,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
        power=2.0
    )

    S_db = librosa.power_to_db(S, ref=np.max)
    S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

    if S_db.shape[1] < T_TARGET:
        S_db = np.pad(S_db, ((0, 0), (0, T_TARGET - S_db.shape[1])), mode="edge")
    else:
        S_db = S_db[:, :T_TARGET]

    return S_db.astype(np.float32)

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)  # stabilize
    ex = np.exp(x)
    return ex / np.sum(ex, axis=1, keepdims=True)

def decide_label(conf):
    max_conf = float(conf.max())
    best_idx = int(conf.argmax())
    second_conf = float(np.sort(conf)[-2])
    entropy = float(-np.sum(conf * np.log(conf + 1e-8)))

    is_undef = (max_conf < THR_CONF) or ((max_conf - second_conf) < THR_MARGIN) or (entropy > THR_ENTROPY)

    if is_undef:
        return "Undefined", b"U", max_conf, second_conf, entropy
    else:
        lab = CLASSES[best_idx]
        cmd = bytes(lab[0], "utf-8")  # H / N / S
        return lab, cmd, max_conf, second_conf, entropy

# ================= INPUT TABS =================
tab1, tab2, tab3 = st.tabs(["🎙️ Record", "📁 Upload", "🎼 Samples"])

# --- Tab 1: Record ---
with tab1:
    rec = st.audio_input("Record your recitation")
    if rec:
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        y, _ = librosa.load(rec, sr=SR, mono=True)
        sf.write(tmp.name, y, SR)
        st.session_state.uploaded_path = tmp.name
        st.session_state.input_source = "record"
        st.audio(tmp.name)  # replay

# --- Tab 2: Upload (WITH REPLAY) ---
with tab2:
    file = st.file_uploader("Upload WAV file", type=["wav"])
    if file:
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        y, _ = librosa.load(file, sr=SR, mono=True)
        sf.write(tmp.name, y, SR)
        st.session_state.uploaded_path = tmp.name
        st.session_state.input_source = "upload"

        st.success("File uploaded.")
        st.audio(tmp.name)  # ✅ replay upload

# --- Tab 3: Samples (KEPT) ---
SAMPLES = {
    "Hijaz": "samples/hijaz.wav",
    "Nahawand": "samples/nahawand.wav",
    "Saba": "samples/saba.wav",
    "Speech": "samples/speech.wav",
    "Noise": "samples/noise.wav",
}

with tab3:
    choice = st.selectbox("Choose a sample:", list(SAMPLES.keys()))
    sample_path = SAMPLES[choice]

    if not os.path.exists(sample_path):
        st.error(f"Sample missing: {sample_path}")
    else:
        st.audio(sample_path)  # ✅ replay sample
        if st.button("Use This Sample"):
            st.session_state.uploaded_path = sample_path
            st.session_state.input_source = f"sample:{choice}"

st.divider()

# ================= RUN PIPELINE =================
if st.session_state.uploaded_path:
    path = st.session_state.uploaded_path

    st.subheader("⚙️ Processing")
    st.write(f"Source: **{st.session_state.input_source}**")
    st.write(f"Audio path: `{path}`")

    mel = wav_to_mel(path)
    mel_in = mel[np.newaxis, np.newaxis, :, :]  # (1,1,128,430)

    logits = session.run(None, {session.get_inputs()[0].name: mel_in})[0]  # (1,3)
    probs = softmax(logits)
    conf = probs[0]

    label, cmd, max_conf, second_conf, entropy = decide_label(conf)

    # --- Send to Arduino ---
    ok, err = send_cmd(cmd) if ensure_serial_ok() else (False, "Arduino not connected")
    if ok:
        st.success(f"LED Output sent to Arduino: {cmd.decode('utf-8')}")
    else:
        st.warning(f"LED Output disabled: {err}")

    # ================= DISPLAY RESULTS =================
    st.subheader("🎼 Prediction Result")
    st.markdown(f"**Detected Category:** `{label}`")
    st.metric("Confidence Score", f"{max_conf*100:.1f} / 100")

    with st.expander("Decision Details"):
        st.write(f"Max confidence: **{max_conf:.3f}**")
        st.write(f"2nd confidence: **{second_conf:.3f}**")
        st.write(f"Margin: **{(max_conf-second_conf):.3f}** (threshold: {THR_MARGIN})")
        st.write(f"Entropy: **{entropy:.3f}** (threshold: {THR_ENTROPY})")
        st.write(f"Rule: Undefined if (conf<{THR_CONF}) OR (margin<{THR_MARGIN}) OR (entropy>{THR_ENTROPY})")

    st.subheader("📊 Class Confidence Distribution")
    st.bar_chart({CLASSES[i]: float(conf[i]) for i in range(3)})

    st.subheader("🔍 Mel-Spectrogram")
    fig, ax = plt.subplots(figsize=(7, 3))
    librosa.display.specshow(mel, sr=SR, x_axis="time", y_axis="mel",
                             fmin=FMIN, fmax=FMAX, cmap="viridis", ax=ax)
    ax.set(title="Mel-Spectrogram")
    st.pyplot(fig)
else:
    st.info("Choose an input (Record / Upload / Sample) to start.")


2026-01-09 01:54:54.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.279 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-01-09 01:54:54.282 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.286 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.286 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.292 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.294 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.295 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-09 01:54:54.297 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar